# Baseline B: Open-VCLIP ViT-B/16 on the full UCF101 test list (Kaggle)

Fourth Kaggle session. Session 3 checked this repository's Open-VCLIP adapter against the
**official** test script on 202 UCF101 videos: top-1 agreed on 200 of them. This session runs
Baseline B on all 3,783 XOV UCF101 test videos (101 classes, 50 familiar / 51 unfamiliar):

1. **Adapter, native configuration** (`openvclip_native_8x3x1`: 8 frames × stride 16 at 30 fps,
   3 views × 1 crop, 28 templates, weight patching 0.5). This produces the reported native
   result: overall, familiar and unfamiliar accuracy with cluster-bootstrap intervals.
2. **Official test script on the same 3,783 videos** (settings of
   `script/testing/temporal_test_b16/ucf.sh`, as in session 3), then `compare-native`: top-1
   agreement on the full list, with every disagreeing video listed.
3. **Adapter, common protocol** (`common_16x1x1`, the same protocol as frozen CLIP in session 1).
   It is reported in its own section and never mixed with native results.

Nothing is tuned: every setting was fixed before this run.

**Before running**

1. Settings → Accelerator: **GPU T4**. Settings → Internet: **On**.
2. Optional input: dataset **lifailifai/ucf101**. Without it, section 3 downloads the official
   archive from crcv.ucf.edu (6.9 GB, about 16 GB of temporary disk).
3. Save Version → **Save & Run All (Commit)**, so the run continues with the browser closed. A
   failing stage is recorded (with the end of its output), and the remaining stages still run.
   The last cell lists every stage's status. Download **`ovar_outputs_4.zip`** from the version's
   Output tab. To repeat only some stages, set the `RUN_*` flags in the first code cell.

Rough time, about 3 hours in total:
* setup 15–20 min, plus 10–20 min if the official UCF101 archive must be downloaded;
* stage 1 about 55 min, stage 2 about 30 min, stage 3 about 65 min.

In [ ]:
import os, subprocess, sys, json, hashlib, shutil
from pathlib import Path

REPO = "eylulpelinkilic/generalized-category-discovery"
REF = "main"            # replace with a commit SHA to pin the code version
WORK = Path("/kaggle/working")
CODE = WORK / "generalized-category-discovery"
PROJECT = CODE / "action_recognition"
RUN_NATIVE = True       # 1. adapter, native configuration, full list (about 55 min)
RUN_OFFICIAL = True     # 2. official test script on the full list + comparison (about 40 min with its environment)
RUN_COMMON = True       # 3. adapter, common protocol, full list (about 65 min)
ZIP = "ovar_outputs_4.zip"
STATUS = {}


def run(cmd, cwd=None, allow_fail=False, log=None):
    """Run a shell command, stream (and optionally save) its output, raise on failure."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or (PROJECT if PROJECT.exists() else WORK),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    sink = open(log, "a") if log else None
    tail = []
    for line in p.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line.rstrip()])[-15:]
        if sink:
            sink.write(line)
    code = p.wait()
    if sink:
        sink.close()
    if code and not allow_fail:
        last = "\n".join(tail)
        raise RuntimeError(f"command failed with exit code {code}: {cmd}\n--- last output lines ---\n{last}")
    return code


def step(name, fn, needs=(), enabled=True):
    """Run one stage; record a failure and continue, so independent later stages still run."""
    missing = [n for n in needs if STATUS.get(n) != "ok"]
    if not enabled:
        STATUS[name] = "not requested"
        return
    if missing:
        STATUS[name] = f"skipped: needs {', '.join(missing)}"
        print(f"-- {name} skipped (needs {missing})")
        return
    try:
        fn()
        STATUS[name] = "ok"
    except Exception as exc:
        STATUS[name] = f"failed: {exc}"
        print(f"!! {name} failed: {exc}")


def sha256(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

## 1. Code: clone the repository

In [ ]:
if subprocess.run(["git", "ls-remote", f"https://github.com/{REPO}", REF],
                  capture_output=True, text=True).returncode != 0:
    raise RuntimeError("Cannot reach GitHub. Turn Settings -> Internet on (phone-verified account), "
                       "then restart the session.")

if not CODE.exists():
    CODE.mkdir(parents=True)
    run("git init -q && git remote add origin https://github.com/" + REPO + ".git", cwd=CODE)
run(f"git fetch -q --depth 1 origin {REF} && git checkout -q FETCH_HEAD", cwd=CODE)
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=CODE, capture_output=True, text=True).stdout.strip()
print("code commit:", COMMIT)
assert (PROJECT / "kaggle" / "ovar_ucf101_openvclip_full.ipynb").exists(), \
    "the cloned code predates this notebook: push the latest commit to GitHub (or set REF)"

## 2. Environment (this repository)

In [ ]:
run("pip install -q open_clip_torch ftfy regex")
if subprocess.run(["which", "ffmpeg"], capture_output=True).returncode != 0:
    run("apt-get -qq update && apt-get -qq install -y ffmpeg")
run("ffmpeg -version | head -1")

import torch, open_clip
print("python", sys.version.split()[0], "| torch", torch.__version__, "| open_clip", open_clip.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Settings -> Accelerator: GPU"

run("python -m ovar fetch-repos")
run("python -m unittest discover -s tests -t .")

## 3. Data: attached Kaggle copy, otherwise the official archive

Same logic as sessions 1–3. The source is written to `runs/DATA_SOURCE.json`.

In [ ]:
OFFICIAL_URL = "https://www.crcv.ucf.edu/data/UCF101/UCF101.rar"
OFFICIAL_BYTES = 6_932_971_618       # Content-Length served by CRCV (Last-Modified 2012-11-21)
DOWNLOAD_DIR = Path("/tmp/ucf101")   # ephemeral disk, not the 20 GB /kaggle/working output


def find_attached():
    found = [p.parent for depth in range(5)
             for p in Path("/kaggle/input").glob("*/" * depth + "UCF-101/ApplyEyeMakeup")]
    return sorted(found, key=lambda p: len(p.parts))


def download_official():
    root = DOWNLOAD_DIR / "UCF-101"
    rar = DOWNLOAD_DIR / "UCF101.rar"
    if (root / "ApplyEyeMakeup").is_dir() and not rar.exists():
        print("already extracted in this session:", root)
        return root, json.loads((DOWNLOAD_DIR / "archive.json").read_text())
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(DOWNLOAD_DIR).free
    if free < 2.3 * OFFICIAL_BYTES:
        raise RuntimeError(f"need about 16 GB free in {DOWNLOAD_DIR}, have {free / 1e9:.1f} GB")

    for attempt in range(3):         # curl resumes a partial file with -C -
        if rar.exists() and rar.stat().st_size == OFFICIAL_BYTES:
            break
        p = subprocess.Popen(["curl", "-sS", "-L", "--fail", "--retry", "5", "--retry-delay", "10",
                              "-C", "-", "-o", str(rar), OFFICIAL_URL])
        while True:
            try:
                p.wait(timeout=60)
                break
            except subprocess.TimeoutExpired:
                size = rar.stat().st_size if rar.exists() else 0
                print(f"  downloaded {size / 1e9:.2f} / {OFFICIAL_BYTES / 1e9:.2f} GB", flush=True)
    size = rar.stat().st_size if rar.exists() else 0
    if size != OFFICIAL_BYTES:
        raise RuntimeError(f"download incomplete: {size} of {OFFICIAL_BYTES} bytes")
    archive = {"url": OFFICIAL_URL, "bytes": size, "sha256": sha256(rar)}
    print("archive:", archive)

    run("apt-get -qq update", allow_fail=True)
    extractors = [("unrar", "unrar", f"unrar x -idq -o+ {rar} {DOWNLOAD_DIR}/"),
                  ("unar", "unar", f"unar -q -f -o {DOWNLOAD_DIR} {rar}"),
                  ("bsdtar", "libarchive-tools", f"bsdtar -xf {rar} -C {DOWNLOAD_DIR}")]
    for tool, package, cmd in extractors:
        if shutil.which(tool) is None:
            run(f"apt-get -qq install -y {package}", allow_fail=True)
        if shutil.which(tool) and run(cmd, allow_fail=True) == 0 and (root / "ApplyEyeMakeup").is_dir():
            archive["extracted_with"] = tool
            break
    else:
        raise RuntimeError("could not extract UCF101.rar (tried unrar, unar, bsdtar)")
    (DOWNLOAD_DIR / "archive.json").write_text(json.dumps(archive) + "\n")
    rar.unlink()
    return root, archive


candidates = find_attached()
if candidates:
    UCF_ROOT = candidates[0]
    DATA_SOURCE = {"source": "kaggle input", "path": str(UCF_ROOT),
                   "other_copies": [str(p) for p in candidates[1:]]}
else:
    print("no attached UCF101 copy under /kaggle/input; downloading the official archive")
    UCF_ROOT, archive = download_official()
    DATA_SOURCE = {"source": "official CRCV archive", "path": str(UCF_ROOT), **archive}
DATA_SOURCE["n_avi"] = sum(1 for _ in UCF_ROOT.glob("*/*.avi"))
print(json.dumps(DATA_SOURCE, indent=2))
(PROJECT / "runs").mkdir(exist_ok=True)
(PROJECT / "runs" / "DATA_SOURCE.json").write_text(json.dumps(DATA_SOURCE, indent=2) + "\n")

CKPT_DIR = Path("/tmp/checkpoints")              # outside /kaggle/working: not part of the outputs
OVC_CKPT = CKPT_DIR / "openvclip-b16" / "swa_2_22.pth"
CLIP_JIT = Path.home() / ".cache" / "clip" / "ViT-B-16.pt"   # where the official code looks for it
(PROJECT / "configs" / "paths.yaml").write_text(
    "dataset_roots:\n"
    f"  ucf101: {UCF_ROOT}\n"
    "hmdb51_single_class_folder: false\n"
    "checkpoints:\n"
    f"  openvclip_vitb16: {OVC_CKPT}\n"
    f"  clip_vitb16_openai_pt: {CLIP_JIT}\n"
    "cache_dir: cache\n"
    "runs_dir: runs\n")
print((PROJECT / "configs" / "paths.yaml").read_text())
run("python -m ovar validate --datasets ucf101 --check-files --check-decode --decode-limit 100")

## 4. Weights

* Open-VCLIP ViT-B/16 `swa_2_22.pth` from the official Google Drive folder (README →
  `openvclip-b16`). No checksum is published; the size is checked and the SHA-256 recorded.
* Original OpenAI `ViT-B-16.pt`, used to build the model and for weight patching
  (`TEST.CLIP_ORI_PATH`). Its URL contains its SHA-256, which is verified.

The SHA-256 of `swa_2_22.pth` must equal session 3's (docs/backbones.md: `49a5bf45…`).

In [ ]:
OVC_URL = ("https://drive.usercontent.google.com/download?id=1QLdkVYPP_AjFJgB9mGNLhppqvHFWkT4x"
           "&export=download&confirm=t")
OVC_BYTES = 598_587_149
CLIP_JIT_SHA = "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f"
CLIP_JIT_URL = f"https://openaipublic.azureedge.net/clip/models/{CLIP_JIT_SHA}/ViT-B-16.pt"
CLIP_JIT_BYTES = 350_837_078
OVC_SHA = "49a5bf452d044f18e8a3c2dff63d46986aeb082be22efe74b70df28cfe08e570"   # session 3
DOWNLOADS = {}


def fetch(url, dest, nbytes):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not (dest.exists() and dest.stat().st_size == nbytes):
        run(f"curl -sS -L --fail --retry 5 -C - -o '{dest}' '{url}'")
    size = dest.stat().st_size
    if size != nbytes:
        raise RuntimeError(f"{dest.name}: got {size} bytes, expected {nbytes}. If Google Drive refuses "
                           "(download quota), download the file in a browser, upload it as a private "
                           f"Kaggle dataset, and copy it to {dest}.")
    return {"url": url, "bytes": size, "sha256": sha256(dest)}


def weights():
    DOWNLOADS["openvclip_vitb16"] = fetch(OVC_URL, OVC_CKPT, OVC_BYTES)
    DOWNLOADS["clip_vitb16_openai_pt"] = fetch(CLIP_JIT_URL, CLIP_JIT, CLIP_JIT_BYTES)
    print(json.dumps(DOWNLOADS, indent=2))
    (PROJECT / "runs" / "DOWNLOADS.json").write_text(json.dumps(DOWNLOADS, indent=2) + "\n")
    if DOWNLOADS["clip_vitb16_openai_pt"]["sha256"] != CLIP_JIT_SHA:
        raise RuntimeError("ViT-B-16.pt SHA-256 does not match its official URL")
    if DOWNLOADS["openvclip_vitb16"]["sha256"] != OVC_SHA:
        raise RuntimeError("swa_2_22.pth differs from the checkpoint checked in session 3")


step("weights", weights)

## 5. Environment for the official Open-VCLIP code (stage 2 only)

Identical to session 3, where it installed and ran the official test script.

The official INSTALL.md uses Python 3.8 and PyTorch 1.11 (CUDA 11.3). This builds that in a separate
virtual environment (`uv` provides Python 3.8), without touching the notebook's own Python.

Two deviations from INSTALL.md, both import-only:
* **detectron2 is replaced by a stub.** The official code imports it at module level for detection
  heads (`ROIAlign`), the demo and visualisation; the classification test path never calls it. Any
  use of the stub raises an error, so it cannot silently change results.
* **The repository is put on `PYTHONPATH`** instead of `python setup.py build develop`: the package is
  pure Python, and its `install_requires` lists `detectron2`, `PIL` and `sklearn`, which pip cannot
  install under those names.

Pinned versions: pytorchvideo from GitHub (as INSTALL.md says) at `eb04d1b`, the state of its main
branch on the date of the pinned Open-VCLIP commit (2024-02-19); PyAV 12.3.0, the last release with
Python 3.8 wheels (INSTALL.md installs an unpinned `av` from conda-forge); protobuf 3.20.3 for
TensorBoard, which the test script imports.

In [ ]:
OVC = PROJECT / "third_party" / "Open-VCLIP"
ENV = Path("/tmp/ovc-env")
PY38 = ENV / "bin" / "python"
STUBS = Path("/tmp/ovc-stubs")
PYTORCHVIDEO_COMMIT = "eb04d1b21e08cfd0713164c0907aeb4c98fd83af"   # main on 2024-02-19
TORCH_INDEX = "--extra-index-url https://download.pytorch.org/whl/cu113"
CONSTRAINTS = Path("/tmp/ovc-constraints.txt")
OVC_PATH = f"PYTHONPATH={STUBS}:{OVC}"

STUB_SOURCE = (
    '"""Import-only stand-in for detectron2 (see the notebook). Using anything from it raises."""\n'
    "class _Unavailable:\n"
    "    def __init__(self, *args, **kwargs):\n"
    "        raise RuntimeError(\"detectron2 is stubbed: this code path needs the real package\")\n\n\n"
    "def __getattr__(name):\n"
    "    return _Unavailable\n")


def official_env():
    CONSTRAINTS.write_text("torch==1.11.0+cu113\ntorchvision==0.12.0+cu113\nnumpy==1.23.5\n"
                           "setuptools==59.5.0\nav==12.3.0\nprotobuf==3.20.3\n")
    run("pip install -q uv")
    run("uv python install 3.8")
    if not PY38.exists():
        run(f"uv venv --python 3.8 --seed {ENV}")
    pip = f"{PY38} -m pip install -q -c {CONSTRAINTS} {TORCH_INDEX}"
    run(f"{pip} setuptools wheel")
    run(f"{pip} torch torchvision")
    run(f"{pip} scipy pandas scikit-learn ftfy regex simplejson psutil opencv-python-headless tensorboard "
        f"iopath fvcore av matplotlib tqdm termcolor yacs pyyaml "
        f"'pytorchvideo @ git+https://github.com/facebookresearch/pytorchvideo.git@{PYTORCHVIDEO_COMMIT}'")
    for rel in ["detectron2/__init__.py", "detectron2/layers.py", "detectron2/config.py",
                "detectron2/engine.py", "detectron2/model_zoo.py", "detectron2/utils/__init__.py",
                "detectron2/utils/visualizer.py"]:
        (STUBS / rel).parent.mkdir(parents=True, exist_ok=True)
        (STUBS / rel).write_text(STUB_SOURCE)
    run(f"{PY38} -c \"import torch, torchvision, av, numpy; print('torch', torch.__version__, "
        f"'torchvision', torchvision.__version__, 'av', av.__version__, 'numpy', numpy.__version__); "
        f"print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0)); torch.zeros(1).cuda()\"")
    run(f"{OVC_PATH} {PY38} -c \"import sys; sys.path.insert(0, 'tools'); import run_net; "
        f"print('official code imports')\"", cwd=OVC)
    run(f"{PY38} -m pip freeze > {PROJECT / 'runs' / 'official_env_freeze.txt'}")


step("official_env", official_env, enabled=RUN_OFFICIAL)

## 6. Stage 1: adapter, native configuration, full UCF101 test list

The checkpoint is a pickle, hence `model.trust_pickle=true`. It comes from the official Drive folder,
and its SHA-256 equals session 3's.

In [ ]:
NATIVE_RUN = PROJECT / "runs" / "openvclip_vitb16__openvclip_native_8x3x1__ucf101"


def adapter_native():
    run("python -m ovar infer openvclip_vitb16__openvclip_native_8x3x1 "
        "--set 'datasets=[ucf101]' --set model.trust_pickle=true --variant ucf101")


step("adapter_native", adapter_native, needs=["weights"], enabled=RUN_NATIVE)

## 7. Stage 2: official test script on the same 3,783 videos, then the comparison

The export writes the full XOV UCF101 test list in `infer`'s order, with the same class texts.
The command is session 3's: only the list changes.

In [ ]:
CHECK = PROJECT / "runs" / "native_check"
EXPORT = CHECK / "export_ovc_full"
OFFICIAL_OUT = CHECK / "official_ovc_full"


def official_full():
    run(f"python -m ovar export-native --backbone openvclip --datasets ucf101 --out {EXPORT}")
    OFFICIAL_OUT.mkdir(parents=True, exist_ok=True)
    run(f"{OVC_PATH} {PY38} -W ignore -u tools/run_net.py "
        "--cfg configs/Kinetics/TemporalCLIP_vitb16_8x16_STAdapter.yaml --opts "
        f"DATA.PATH_TO_DATA_DIR {EXPORT / 'ucf101'} DATA.PATH_PREFIX {UCF_ROOT} DATA.PATH_LABEL_SEPARATOR , "
        f"DATA.INDEX_LABEL_MAPPING_FILE {EXPORT / 'ucf101' / 'ucf101-index2cls.json'} "
        f"TRAIN.ENABLE False OUTPUT_DIR {OFFICIAL_OUT} TEST.BATCH_SIZE 8 NUM_GPUS 1 "
        "DATA.DECODING_BACKEND pyav MODEL.NUM_CLASSES 101 DATA_LOADER.NUM_WORKERS 4 "
        f"TEST.CUSTOM_LOAD True TEST.CUSTOM_LOAD_FILE {OVC_CKPT} TEST.SAVE_RESULTS_PATH preds.pkl "
        "TEST.NUM_ENSEMBLE_VIEWS 3 TEST.NUM_SPATIAL_CROPS 1 "
        f"TEST.PATCHING_MODEL True TEST.PATCHING_RATIO 0.5 TEST.CLIP_ORI_PATH {CLIP_JIT}",
        cwd=OVC, log=OFFICIAL_OUT / "console.log")
    assert (OFFICIAL_OUT / "preds.pkl").exists(), "official script finished without writing preds.pkl"


def compare_full():
    run(f"python -m ovar compare-native --official {OFFICIAL_OUT / 'preds.pkl'} --run-dir {NATIVE_RUN} "
        f"--dataset ucf101 --trust-pickle --out {CHECK / 'compare_ucf101_full.json'}")


step("official_full", official_full, needs=["weights", "official_env"], enabled=RUN_OFFICIAL)
step("compare_full", compare_full, needs=["official_full", "adapter_native"], enabled=RUN_OFFICIAL)

## 8. Stage 3: adapter, common protocol, full UCF101 test list

Same protocol as frozen CLIP in session 1: 16 segment-midpoint frames, 1 view, CLIP's centre
crop, the prompt "a video of a person {}.". Open-VCLIP was trained with 8 frames; the official
model code builds its temporal attention for any frame count.

In [ ]:
def adapter_common():
    run("python -m ovar infer openvclip_vitb16__common_16x1x1 "
        "--set 'datasets=[ucf101]' --set model.trust_pickle=true --variant ucf101")


step("adapter_common", adapter_common, needs=["weights"], enabled=RUN_COMMON)

## 9. Reports and outputs

In [ ]:
step("report", lambda: run("python -m ovar report"))
if STATUS.get("report") == "ok":
    from IPython.display import Markdown, display
    display(Markdown((PROJECT / "reports" / "results.md").read_text()))
if (CHECK / "compare_ucf101_full.json").exists():
    result = json.loads((CHECK / "compare_ucf101_full.json").read_text())
    print("native check, full list:", {k: v for k, v in result.items() if k != "disagreements"},
          f"| {len(result['disagreements'])} disagreeing videos")

(PROJECT / "runs" / "CODE_COMMIT.txt").write_text(COMMIT + "\n")
(PROJECT / "runs" / "NOTEBOOK_STATUS.json").write_text(json.dumps(STATUS, indent=2) + "\n")
run(f"rm -f {WORK}/{ZIP} && cd {PROJECT} && zip -qr {WORK}/{ZIP} runs reports configs/paths.yaml")
run(f"ls -lh {WORK}/{ZIP}")

In [ ]:
# Does not raise: an error here could cost the saved outputs of a committed run.
failed = {k: v for k, v in STATUS.items() if v not in ("ok", "not requested")}
print("=" * 70)
print("STAGES NOT COMPLETED - send the zip or this output:" if failed else "all requested stages completed")
for name, state in STATUS.items():
    print(f"{name:22s} {state}")
print("=" * 70)